### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

/Users/francomorero/PycharmProjects/franco/ceia-trabajos/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP, que ya viene incluido y formateado en sklearn.

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [5]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [6]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [7]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [8]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [9]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palabra que no está en el documento.

In [10]:
# tfidfvect.vocabulary_['cocoliso']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [11]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [12]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [13]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [14]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [15]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [16]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [17]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750])

Obtenemos los 5 documentos más similares:

In [18]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [19]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [20]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [21]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ajustado en train, así que vectorizamos los textos
del conjunto de test.

In [22]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [23]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.


In [24]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

def clase(i):
    return newsgroups_train.target_names[y_train[i]]

def texto(i, chars=300):
    """Documento i sin saltos de linea, recortado a chars."""
    return " ".join(newsgroups_train.data[i].split())[:chars]

def top_tokens(i, n_tokens=10):
    """Tokens de mayor peso tf-idf del documento i."""
    fila = X_train[i].tocoo()
    vocab = tfidfvect.get_feature_names_out()
    orden = np.argsort(fila.data)[::-1][:n_tokens]
    return [(vocab[fila.col[j]], round(float(fila.data[j]), 3)) for j in orden]

def similares(idx, n=5):
    """Indices y similitudes de los n documentos mas parecidos a idx."""
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    top = np.argsort(cossim)[::-1][1:n+1]
    return top, cossim[top]

def tabla_similares(idx, n=5, chars=300, n_tokens=10):
    """Imprime el documento consulta y devuelve la tabla de sus n mas similares."""
    print(f"CONSULTA idx={idx} | clase={clase(idx)}")
    print(texto(idx, chars))
    print("tokens tf-idf:", ", ".join(f"{t}({p})" for t, p in top_tokens(idx, n_tokens)), "\n")
    top, sims = similares(idx, n)
    return pd.DataFrame({
        "rank": range(1, n + 1),
        "idx": top,
        "similitud": sims.round(4),
        "clase": [clase(i) for i in top],
        "misma_clase": [y_train[i] == y_train[idx] for i in top],
        "tokens": [", ".join(t for t, _ in top_tokens(i, n_tokens)) for i in top],
        "texto": [texto(i, chars) for i in top],
    })

In [25]:
idxs = [5431, 142, 2734, 7467, 8129]

In [26]:
tabla_similares(idxs[0])


CONSULTA idx=5431 | clase=talk.politics.guns
I phoned Licensing Division in Washington State to ask for an application for a CCW. Instead they promptly sent me an applicationfor becoming a firearms dealer in Washington!
tokens tf-idf: washington(0.38), applicationfor(0.361), phoned(0.304), ccw(0.294), promptly(0.278), licensing(0.266), becoming(0.225), firearms(0.224), dealer(0.211), division(0.203) 



,rank,idx,similitud,clase,misma_clase,tokens,texto
0,1,5784,0.1607,rec.motorcycles,False,"motorcycle, italy, washington, displacement, state, insured, endorsement, instructor, italian, unlimited","Anyone in Europe got any advice for a US citizen whose going to be living and working in Italy for a year and wants to buy a motorcycle there? An Italian friend just arrived here in Washington State to work for two years, and she's finding it very very difficult to obtain car insurance. So I thought"
1,2,1324,0.1588,rec.autos,False,"dealer, cost, invoice, incentives, factory, believing, necessarily, trouble, the, sell","is this really the dealer's cost? did you get the dealer's cost by looking at the invoice? there may be factory to dealer incentives. i'd check this out, since i have trouble believing that a dealer would sell a car to me at his cost. dealer invoice is not necessarily the dealer cost."
2,3,6674,0.1341,comp.sys.mac.hardware,False,"tapedrive, dealer, quadra, our, anybody, inits, 8500, exabyte, we, phoned","I would like more info on this if anybody has it. Our Exabyte 8500 tapedrive has never been working from the Quadra 950. We have been trying it since September 1992, replaced cabling, inits, I don't know what all. All the ""industry experts"" we phoned (the tapedrive dealer, our Apple dealer, the soft"
3,4,1263,0.1257,sci.space,False,"washington, nw, 202, 212, street, dc, york, ny, 20036, news","Early to mid June. If they think the public wants to see it they will carry it. Why not write them and ask? You can reach them at: F: NATIONAL NEWS MEDIA ABC ""World News Tonight"" ""Face the Nation"" 7 West 66th Street CBS News New York, NY 10023 2020 M Street, NW 212/887-4040 Washington, DC 20036 202/"
4,5,8605,0.1177,comp.graphics,False,"dxf, stored, ccw, cw, polygons, points, polygon, order, program, 3d",Hi. I'm writing a program to convert .dxf files to a database format used by a 3D graphics program I've written. My program stores the points of a polygon in CCW order. I've used 3D Concepts a little and it seems that the points are stored in the order they are drawn. Does the DXF format have a way


Veo que ninguno de los 5 más similares cae en talk.politics.guns y que las similitudes son bajísimas, entre 0.11 y 0.16. Mirando la columna de tokens podemos ver que el rank 1 comparte washington pero es un post de motos; el rank 4 también comparte washington, pero son direcciones de canales de TV en Washington DC; y el rank 5 comparte ccw, pero no es un permiso de armas sino que viene de un problema de polígonos. En el rank 2 parece que le dio peso a dealer y en el rank 3 a dealer y phoned, pero tampoco tienen relación con el texto original.

In [27]:
tabla_similares(idxs[1])


CONSULTA idx=142 | clase=comp.os.ms-windows.misc
Version 2.03 drivers are current.
tokens tf-idf: 03(0.618), drivers(0.484), current(0.419), version(0.417), are(0.185) 



,rank,idx,similitud,clase,misma_clase,tokens,texto
0,1,10853,0.3885,rec.sport.baseball,False,"03, 02, 04, 01, 05, lost, won, 06, 07, 556","MLB Standings and Scores for Satruday, April 17th, 1993 (including yesterday's games) NATIONAL WEST Won Lost Pct. GB Last 10 Streak Home Road San Francisco Giants 07 04 .636 -- 6-4 Won 2 04-01 03-03 Houston Astros 06 04 .600 0.5 6-4 Won 1 01-03 05-01 Atlanta Braves 06 06 .500 1.5 5-5 Lost 3 04-03 03"
1,2,3530,0.3280,rec.sport.baseball,False,"02, 03, 04, 01, 00, 05, lost, won, 06, 571","MLB Standings and Scores for Thursday, April 15th, 1993 (including yesterday's games) NATIONAL WEST Won Lost Pct. GB Last 10 Streak Home Road Houston Astros 05 03 .625 -- 5-3 Won 5 00-03 05-00 Atlanta Braves 06 04 .600 -- 6-4 Lost 1 03-03 03-01 San Francisco Giants 05 04 .556 0.5 5-4 Lost 1 02-01 03"
2,3,5578,0.3278,comp.os.ms-windows.misc,True,"24x, drivers, call, unaware, fixes, problem, indicated, 03, diamond, failure","I had the exact same failure with the 24X and Word for Windows. A quick call to Microsoft indicated it was problem with the 24X drivers. You need to call Diamond and get the new drivers, I think version 2.03 fixes the above problem, there may be later versions that I'm unaware of..."
3,4,7812,0.3110,comp.os.ms-windows.misc,True,"drivers, gpf, gateway, problems, the, have, with, jumpy, attributable, abusing","I have used both version 1.17 drivers for Win 3.1 and the new 2.03 drivers. I have had none of these problems. No GPF's at all. I have a feeling that your problems are not with the card or drivers. The ATI Ultra drivers are considered some of the most reliable on the market, and the SS 24X ones seem"
4,5,2356,0.3010,comp.os.ms-windows.misc,True,"version, ghostscript, satisfied, plain, 386, versions, dos, quite, windows, actually","I've been using version 2.5.2 of ghostscript, and I'm quite satisfied with it. There are, actually, 3 versions: a plain dos version, a 386 version, and a windows version."


El documento original son cinco palabras. Vemos que los dos primeros son tablas de posiciones de la MLB y podemos ver en sus tokens que aparecen 03, 02, 04 y 01, que son los resultados de los partidos y coinciden con el 03 de la versión de los drivers del documento de windows. Del rank 3 al 5 sí aparecen posts de Windows con similitud sobre drivers, motivados por los tokens 24x, drivers, 03, y version, que coinciden un poco más con el documento original. Lo notorio es cómo estos documentos quedan por debajo de los de baseball, porque TF-IDF le dio más peso al token 03.

In [28]:
tabla_similares(idxs[2])


CONSULTA idx=2734 | clase=comp.sys.mac.hardware
I remember reading a thread a few days ago that mentioned removing an external syquest drive from its case and dropping it in the internal drive of a Centris. . . I was going to do that with my 610, but had a couple of questions. My PLI 80M syquest drive has a wire from the drive to an id# switch on
tokens tf-idf: drive(0.428), the(0.278), panel(0.266), syquest(0.225), motherboard(0.168), case(0.163), switch(0.158), internal(0.153), spotsbm(0.142), tmpty(0.142) 



,rank,idx,similitud,clase,misma_clase,tokens,texto
0,1,3725,0.3751,comp.sys.ibm.pc.hardware,False,"drive, scsi, fdisk, the, partitions, seagate, formatted, mg, and, is","I have a 486sx25 computer with a 105 Mg Seagate IDE drive and a controler built into the motherboard. I want to add a SCSI drive (a quantum prodrive 425F 425 MG formatted). I have no documentation at all and I need your help! As I understand it, here is the process of adding such a drive. Could you"
1,2,6764,0.3539,comp.sys.ibm.pc.hardware,False,"bootable, drive, boot, switch, drives, the, install, disks, chasis, interchanged",": I have a 5 1/4"" drive as drive A. How can I make the system boot from : my 3 1/2"" B drive? (Optimally, the computer would be able to boot : from either A or B, checking them in order for a bootable disk. But : if I have to switch cables around and simply switch the drives so that : it can't boot 5"
2,3,3035,0.3337,comp.sys.ibm.pc.hardware,False,"drive, the, drives, booted, disk, cmos, switched, si, lights, norton","Hi! I have a problem with my floppy drives. In an effort to make my 3.5"" drive (normally b:) my a: drive, I switched the order of connections on the cable from the serial card/floppy/ide controller. I booted up, changed the CMOS settings to reflect the a: drive as the 3.5 and the b: drive as the 5.2"
3,4,6386,0.3181,comp.sys.ibm.pc.hardware,False,"drive, the, slave, master, cyl, western, 977, digital, currently, formatted","I was recently loaned an older Dec 210 286 at work, and I have the option of adding an additional Western Digital Hard-drive to the machine. The existing drive is currently a Western Digital as well, and is working fine, but I do not have any documentation available for configuring the master/slave"
4,5,1524,0.3102,comp.sys.mac.hardware,True,"drives, syquest, drive, massmicro, macleak, pli, the, cartridges, are, 105","I think you must be talking about the Syquest 105 (code named Mesa I believe). It is a 3.5"" Winchester technology drive pretty much like the other Syquest drives in terms of how it works. According to the latest MacLeak, the drive has a 14.5 ms access time, 1.9 MB/s sustained throughput (these figur"


En este caso la similitud tiene sentido aunque la etiqueta no coincida. 4 de los 5 son comp.sys.ibm.pc.hardware y sus tokens son todos del mismo campo, drive, switch, boot, master y slave, igual que el documento original. Considero que el error es razonable, porque mac y ibm.pc comparten casi todo el léxico técnico y lo único que las separa es el nombre del equipo. De hecho el único de la misma clase, el rank 5, es el que comparte syquest, que es la utilidad de mac mencionada en el documento (https://www.macintoshrepository.org/2206-syquest-utilities-4-0-1).

In [29]:
tabla_similares(idxs[3])


CONSULTA idx=7467 | clase=comp.os.ms-windows.misc
Anyone have any info. on the video/sound card from SIGMA designs. It is called WIN STORM PC. They also have another card called the legend 24lx any info would be appreciated, incuding performance, pricing and availability. thanks
tokens tf-idf: incuding(0.313), 24lx(0.3), info(0.288), card(0.279), sigma(0.272), called(0.256), legend(0.24), pricing(0.238), storm(0.226), designs(0.216) 



,rank,idx,similitud,clase,misma_clase,tokens,texto
0,1,3828,0.6808,comp.os.ms-windows.misc,True,"model, 24lx, sigma, legend, encountered, compatibility, storm, designs, performance, sound","Does anyone out there use a SIGMA designs VIDEO/SOUND card ? The model is called WIN-STORM-PC . They also have one model the Legend-24lx Any info on these like performance and compatibility, or even problems encountered will be appreciated."
1,2,8571,0.2716,soc.religion.christian,False,"appreciated, info, hi, group, anything, thanks, anyone, does, know, any","Hi, Does anyone know anything about this group and what they do? Any info would be appreciated. Thanks!"
2,3,10776,0.2345,comp.os.ms-windows.misc,True,"info, comdex, peers, any, anyone, direction, fall, coming, 93, appreciated","Does anyone out there have any info on the up and coming fall comdex '93? I was asked by one of my peers to get any info that might be available. Or, could anyone point me in the right direction? Any help would be appreciated."
3,4,765,0.2194,comp.sys.mac.hardware,False,"card, mac, the, color, video, 1250, 1738, freeing, it, software","I have a Radius Precision Color 24x video card for the Mac that fits in a NuBus slot. The card has 3 Mb of VRAM on it, which means that 24-bit color is possible on the card! The card supports just about any monitor scan rate you can think of (I used it at 640x480, 800x600 and 1024x768, but it can go"
4,5,270,0.2067,comp.sys.ibm.pc.hardware,False,"card, video, advise, must, deliver, advertising, vendor, need, rich, am","Hi...I need some info on video card. I am looking a video card that can deliver a high quality picture. I need the card to display images (well for advertising company btw), so it must be rich with colors and the speed must be fast too. I am just wondering if somebody can advise me what to buy for s"


El rank 1 es muy similar, se pregunta por SIGMA, por la misma placa WIN-STORM-PC y por legend 24lx, y comparte tokens como sigma, storm, legend y 24lx con 0.68 de similitud. Después la similitud cae a 0.27 con un documento de soc.religion.christian cuyos tokens son appreciated, info, hi, thanks y anyone, que no comparte el tema sino la forma en que se formuló la pregunta original. Los ranks 4 y 5 sí tienen que ver con video card, por lo que el tema tiene cierta similitud, pero las clases son diferentes.

In [30]:
tabla_similares(idxs[4])


CONSULTA idx=8129 | clase=talk.politics.mideast
ac = In <9304202017@zuma.UUCP> sera@zuma.UUCP (Serdar Argic) pl = linden@positive.Eng.Sun.COM (Peter van der Linden) pl: 1. So, did the Turks kill the Armenians? ac: So, did the Jews kill the Germans? ac: You even make Armenians laugh. ac: "An appropriate analogy with the Jewish Holocaust might be t
tokens tf-idf: ac(0.562), the(0.179), armenians(0.176), zuma(0.173), linden(0.159), of(0.141), pl(0.14), did(0.135), ai(0.135), republic(0.127) 



,rank,idx,similitud,clase,misma_clase,tokens,texto
0,1,8129,1.0000,talk.politics.mideast,True,"ac, the, armenians, zuma, linden, of, pl, did, ai, republic","ac = In <9304202017@zuma.UUCP> sera@zuma.UUCP (Serdar Argic) pl = linden@positive.Eng.Sun.COM (Peter van der Linden) pl: 1. So, did the Turks kill the Armenians? ac: So, did the Jews kill the Germans? ac: You even make Armenians laugh. ac: ""An appropriate analogy with the Jewish Holocaust might be t"
1,2,3638,0.2918,sci.electronics,False,"ac, ups, to, the, switchover, mains, battery, system, it, power","Actually, it's a bit more complicated than that...I sounds to me, your UPS takes in AC, rectifies it to DC to charge the batteries, and then takes the battery DC and chops it to AC again, feeding your equipment. This approach is the easiest and cleanest way to switchover from the mains to battery on"
2,3,10912,0.2840,talk.politics.mideast,True,"armenians, muslim, population, slaughter, the, of, armenian, did, been, ottoman","Typical 'Arromdian' of the ASALA/SDPA/ARF Terrorism and Revisionism Triangle. Well, does it change the fact that during the period of 1914 to 1920, the Armenian Government ordered, incited, assisted and participated in the genocide of 2.5 million Muslim people because of race, religion and national"
3,4,70,0.2801,talk.politics.mideast,True,"the, armenian, russian, and, of, armenians, to, genocide, in, ottoman",": Pardon me? Here is to an amherst-clown: : : ""Your three chiefs, Dro, Hamazasp and Kulkhandanian are the ringleaders : of the bands which have destroyed Tartar villages and have staged : massacres in Zangezour, Surmali, Etchmiadzin, and Zangibasar. This is : intolerable. Were you expecting a differ"
4,5,5423,0.2619,talk.politics.mideast,True,"the, of, armenian, constitution, ottoman, russian, and, to, parliament, turkish","Anytime. Suffering from a severe case of myopia? No Muslim left alive - not a single one. Leading the first Armenian units who crossed the Ottoman border in the company of the Russian invaders was the former Ottoman Parliamentary representative for Erzurum, Karekin Pastirmaciyan, who assumed the rev"


El rank 1 tiene similitud 1.0000 porque el documento está duplicado exacto en el dataset, así que el slice que descarta la primera posición saca una copia y queda la otra. Los otros tres de talk.politics.mideast aciertan la clase, y comparten los tokens armenian y "the". Es interesante analizar que acá quizás valdría la pena eliminar el token "the" junto con otras stopwords, para ver si la similitud se mantiene, siendo que los textos nombran armenian pero en temáticas diferentes. El que está mal es el rank 2, que tiene la clase sci.electronics y entra por "ac", que en el documento original son iniciales y en el otro significa corriente alterna.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.
